# EGU26 Short Course: Machine-Learning Downscaling with `idownscale`

This notebook is both:

- a readable scientific and technical walkthrough
- a practical notebook that can be re-executed phase by phase

It is designed to remain usable even for readers who did not attend the short
course. The companion Markdown pages provide more detailed reference material,
but this notebook should contain the full runnable workflow experience.

The intended sequence is:

1. set up the Python environment
2. prepare directories for raw inputs and generated outputs
3. perform the France pre-Phase-1 preparation step
4. run each workflow phase independently
5. inspect statistics, tables, and plots after each phase
6. optionally reuse published outputs when a live rerun would take too long

Some full phases can take several hours. Because of that, the notebook is
designed so that users can stop after any phase, inspect results, and resume
later.


## How To Use This Notebook

Each phase should be treated as an independent block.

For each phase, the notebook should provide:

1. scientific motivation
2. technical explanation
3. the command or commands to run
4. one or more checks showing whether the phase succeeded
5. a short discussion of the outputs

When a live rerun is too long, the notebook can also load already published
Mercure outputs for discussion.

The notebook is the main hands-on experience. The companion Markdown pages give
more detail on installation, data retrieval, workflow phases, expected outputs,
and validation checks.


## 1. Environment Setup

A Conda or Mamba environment is recommended because `xesmf` depends on `ESMF` and `esmpy`.

Typical local installation sequence:

```bash
conda create -n idownscale -c conda-forge python=3.11
conda activate idownscale
conda install -c conda-forge \
  eigen esmpy xesmf netcdf4 cartopy \
  numpy scipy pandas xarray \
  matplotlib seaborn pyproj \
  pytorch torchvision torchaudio pytorch-lightning torchmetrics \
  timm tqdm
export ESMFMKFILE="$CONDA_PREFIX/lib/esmf.mk"
pip install ibicus==1.1.1 monai==1.4.0 SBCK==1.4.2
pip install -e .
```

Known practical issues:

- `xesmf` may fail if `ESMF`, `esmpy`, or `ESMFMKFILE` are not set correctly
- on HPC systems, `PYTHONHOME` and `PYTHONPATH` can interfere with a Conda environment
- `SBCK` is a known extra dependency, but the main validated bias-correction path currently uses `ibicus`

Before starting the workflow, import verification should succeed.

In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
COURSE_MERCURE_URL = "https://mercure.cerfacs.fr/egu26scml/"

# Optional local mirror of the Mercure content.
COURSE_ASSETS_DIR = Path(os.environ.get("IDOWNSCALE_COURSE_ASSETS_DIR", REPO_ROOT / "egu26_course_assets"))

# Main runtime directories.
RAWDATA = Path(os.environ.get("IDOWNSCALE_RAW_DIR", REPO_ROOT / "rawdata"))
OUTPUT_DIR = Path(os.environ.get("IDOWNSCALE_OUTPUT_DIR", REPO_ROOT / "idownscale_output"))
WEIGHTS_DIR = Path(os.environ.get("IDOWNSCALE_REGRID_WEIGHTS_DIR", OUTPUT_DIR / "weights"))
RUNS_DIR = Path(os.environ.get("IDOWNSCALE_RUNS_DIR", OUTPUT_DIR / "runs"))
PREDICTION_DIR = Path(os.environ.get("IDOWNSCALE_PREDICTION_DIR", OUTPUT_DIR / "prediction"))
METRICS_DIR = Path(os.environ.get("IDOWNSCALE_METRICS_DIR", OUTPUT_DIR / "metrics"))
SCRATCH_DIR = Path(os.environ.get("IDOWNSCALE_SCRATCH_DIR", REPO_ROOT / "scratch"))
CHECKPOINT_BUNDLES_DIR = Path(
    os.environ.get("IDOWNSCALE_CHECKPOINT_BUNDLES_DIR", SCRATCH_DIR / "checkpoint_bundles")
)

RAWDATA, OUTPUT_DIR, COURSE_ASSETS_DIR, CHECKPOINT_BUNDLES_DIR


In [ ]:
# Minimal import check.
try:
    import xesmf  # noqa: F401
    import ibicus  # noqa: F401
    import cartopy  # noqa: F401
    import torch  # noqa: F401
    print("Environment import check: OK")
except Exception as exc:
    print("Environment import check failed:")
    print(exc)
    print("Check Conda, esmpy/xesmf, and ESMFMKFILE before continuing.")

## 2. Prepare Directories

The workflow uses a clear separation between raw inputs and generated outputs.

Main path variables:

- `IDOWNSCALE_RAW_DIR`
- `IDOWNSCALE_OUTPUT_DIR`
- `IDOWNSCALE_REGRID_WEIGHTS_DIR`
- `IDOWNSCALE_RUNS_DIR`
- `IDOWNSCALE_PREDICTION_DIR`
- `IDOWNSCALE_METRICS_DIR`
- `IDOWNSCALE_CHECKPOINT_BUNDLES_DIR`

The checkpoint bundles are kept under `scratch/checkpoint_bundles/` in the
published quickstart so that pretrained artifacts stay separate from generated
outputs.

This should be the first practical setup step for users who want to rerun the
workflow on their own machine.


In [ ]:
for path in [
    RAWDATA,
    OUTPUT_DIR,
    WEIGHTS_DIR,
    RUNS_DIR,
    PREDICTION_DIR,
    METRICS_DIR,
    COURSE_ASSETS_DIR,
    SCRATCH_DIR,
    CHECKPOINT_BUNDLES_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

for key, value in {
    "IDOWNSCALE_RAW_DIR": RAWDATA,
    "IDOWNSCALE_OUTPUT_DIR": OUTPUT_DIR,
    "IDOWNSCALE_REGRID_WEIGHTS_DIR": WEIGHTS_DIR,
    "IDOWNSCALE_RUNS_DIR": RUNS_DIR,
    "IDOWNSCALE_PREDICTION_DIR": PREDICTION_DIR,
    "IDOWNSCALE_METRICS_DIR": METRICS_DIR,
    "IDOWNSCALE_CHECKPOINT_BUNDLES_DIR": CHECKPOINT_BUNDLES_DIR,
}.items():
    os.environ[key] = str(value)

print("Configured runtime paths:")
for key in [
    "IDOWNSCALE_RAW_DIR",
    "IDOWNSCALE_OUTPUT_DIR",
    "IDOWNSCALE_REGRID_WEIGHTS_DIR",
    "IDOWNSCALE_RUNS_DIR",
    "IDOWNSCALE_PREDICTION_DIR",
    "IDOWNSCALE_METRICS_DIR",
    "IDOWNSCALE_CHECKPOINT_BUNDLES_DIR",
]:
    print(f"{key}={os.environ[key]}")


## 3. Helper Utilities Used In This Notebook

The next cell defines small helpers to:

- print shell commands clearly
- execute shell commands when requested
- inspect JSON, CSV, image, NetCDF, and NPZ outputs
- reuse already published Mercure outputs when available locally

By default, workflow commands are shown but not executed. Set the execution
toggle to `True` when you want the notebook cells to run the commands directly.


## Important Execution Toggle

By default, the workflow command cells in this notebook only print the commands.

To actually execute the workflow from the notebook:

1. set `EXECUTE_WORKFLOW_COMMANDS = True` in the next helper cell
2. rerun that helper cell
3. then run the phase command cells

This default is intentional so users can read the notebook safely before launching long-running phases.


In [ ]:
import json
import subprocess
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

EXECUTE_WORKFLOW_COMMANDS = False

def show_command(cmd: str) -> None:
    print(cmd)

def run_shell(cmd: str, execute: bool = False) -> None:
    print(cmd)
    if execute:
        subprocess.run(cmd, shell=True, check=True, cwd=REPO_ROOT)

def show_json_table(path: Path, max_rows: int = 20) -> None:
    if not path.exists():
        print(f"Missing file: {path}")
        return
    with path.open() as f:
        data = json.load(f)
    rows = []
    for key, value in data.items():
        if isinstance(value, dict):
            row = {"name": key}
            row.update(value)
            rows.append(row)
        else:
            rows.append({"name": key, "value": value})
    display(pd.DataFrame(rows).head(max_rows))

def show_csv(path: Path, n: int = 10) -> None:
    if not path.exists():
        print(f"Missing file: {path}")
        return
    display(pd.read_csv(path).head(n))

def show_image(path: Path) -> None:
    if not path.exists():
        print(f"Missing image: {path}")
        return
    display(Image(filename=str(path)))

def describe_dataset(path: Path) -> None:
    if not path.exists():
        print(f"Missing dataset: {path}")
        return
    ds = xr.open_dataset(path)
    print(ds)

def describe_npz(path: Path) -> None:
    if not path.exists():
        print(f"Missing NPZ: {path}")
        return
    arr = np.load(path)
    for key in arr.files:
        value = arr[key]
        print(key, value.shape, np.nanmin(value), np.nanmax(value))

print(f"EXECUTE_WORKFLOW_COMMANDS={EXECUTE_WORKFLOW_COMMANDS}")
print("Set EXECUTE_WORKFLOW_COMMANDS=True in this cell before rerunning it to execute workflow commands from later cells.")


## 4. Mercure Course Package

Published short-course material is hosted at:

- `https://mercure.cerfacs.fr/egu26scml/`

Key subdirectories:

- `required/`
- `nice_to_have/`
- `raw_data/`
- `phase_outputs/`
- `egu26_sc_required.tar.gz`
- `egu26_sc_nice_to_have.tar.gz`

This notebook can use those published outputs in two ways:

- as reference results to compare with a live run
- as ready-made figures, tables, and statistics when a live rerun would take too long

The two `.tar.gz` files are the easiest bulk-download route for users who want to retrieve the published assets quickly.

### Example bulk-download commands

Users can download the packaged archives directly from Mercure:

```bash
curl -O https://mercure.cerfacs.fr/egu26scml/egu26_sc_required.tar.gz
curl -O https://mercure.cerfacs.fr/egu26scml/egu26_sc_nice_to_have.tar.gz
tar -xzf egu26_sc_required.tar.gz
tar -xzf egu26_sc_nice_to_have.tar.gz
```

This is often simpler than downloading many individual files one by one.

## 5. Pre-Phase-1 France Preparation

Scientific role:

- prepare the observational target side over France
- make the later workflow consistent with the `exp5` setup

Technical role:

- standardize dimensions and coordinates if needed
- crop E-OBS target-side files to the France domain

This phase should be run independently and verified before moving on.

### Before Running The First Workflow Phase

If you want the next command cells to execute instead of only being printed, make sure you already set `EXECUTE_WORKFLOW_COMMANDS = True` and reran the helper cell above.


In [ ]:
crop_tas_cmd = """
python bin/preprocessing/crop_domain.py \
  --input rawdata/eobs/tas_ens_mean_1d_025deg_reg_v29_0e_19500101-20231231.nc \
  --output rawdata/eobs/tas_ens_mean_1d_025deg_reg_v29_0e_19500101-20231231_france.nc \
  --exp exp5 \
  --standardize
""".strip()

crop_orog_cmd = """
python bin/preprocessing/crop_domain.py \
  --input rawdata/eobs/elevation_ens_025deg_reg_v29_0e.nc \
  --output rawdata/eobs/elevation_ens_025deg_reg_v29_0e_france.nc \
  --exp exp5 \
  --standardize
""".strip()

run_shell(crop_tas_cmd, execute=EXECUTE_WORKFLOW_COMMANDS)
print()
run_shell(crop_orog_cmd, execute=EXECUTE_WORKFLOW_COMMANDS)


In [ ]:
# Validation checks after France preparation.
cropped_tas = RAWDATA / "eobs" / "tas_ens_mean_1d_025deg_reg_v29_0e_19500101-20231231_france.nc"
cropped_orog = RAWDATA / "eobs" / "elevation_ens_025deg_reg_v29_0e_france.nc"

describe_dataset(cropped_tas)
describe_dataset(cropped_orog)

Suggested discussion after this phase:

- Are the longitude and latitude ranges consistent with the France domain?
- Are the dimensions consistent with the expected `exp5` target grid?
- Do the cropped fields look scientifically reasonable?

## 6. Phase 1: Build Daily Samples

Scientific role:

- assemble predictor and target information into daily learning samples

Technical role:

- create `.npz` sample files
- support reduced laptop-scale runs with shorter date windows

This can be a long phase in a full run, so it should be easy to run independently and resume later.

In [ ]:
phase1_cmd_small = """
python bin/production/run_exp5_workflow.py \
  --exp exp5 \
  --steps phase1 \
  --phase1-start-date 19850101 \
  --phase1-end-date 19850103
""".strip()

phase1_cmd_full = """
python bin/production/run_exp5_workflow.py \
  --exp exp5 \
  --steps phase1
""".strip()

run_shell(phase1_cmd_small, execute=EXECUTE_WORKFLOW_COMMANDS)
print()
run_shell(phase1_cmd_full, execute=False)


In [ ]:
# Validation checks after Phase 1.
sample_file = OUTPUT_DIR / "datasets" / "dataset_exp5_30y" / "sample_19850101.npz"
describe_npz(sample_file)

Suggested discussion after this phase:

- What do the predictor and target arrays represent scientifically?
- Are the shapes and value ranges reasonable?
- Are reduced windows sufficient for a laptop demonstration while preserving the workflow logic?

## 7. Statistics Phase

Scientific role:

- summarize the sample distributions before training

Technical role:

- compute `statistics.json`
- produce histogram diagnostics

These statistics and plots can either be generated live or loaded from the already published outputs.

In [ ]:
stats_cmd = """
python bin/production/run_exp5_workflow.py \
  --exp exp5 \
  --steps stats
""".strip()

run_shell(stats_cmd, execute=EXECUTE_WORKFLOW_COMMANDS)


In [ ]:
stats_json = OUTPUT_DIR / "datasets" / "dataset_exp5_30y" / "statistics.json"
show_json_table(stats_json)

for name in ["hist_y_train.png", "hist_y_val.png", "hist_y_test.png"]:
    show_image(OUTPUT_DIR / "datasets" / "dataset_exp5_30y" / name)

Suggested discussion after this phase:

- Do the summary statistics look plausible?
- Do the distributions suggest obvious pathologies?
- How do these statistics connect to later normalization and training behavior?

## 8. Bias-Correction Phases

Scientific role:

- reduce systematic coarse-model bias before using the simulation data as inputs

Technical role:

- build the bias-correction datasets
- apply correction to the coarse GCM inputs

These phases should be discussed separately from the later machine-learning phases.

In [ ]:
bc_dataset_cmd = """
python bin/production/run_exp5_workflow.py \
  --exp exp5 \
  --steps bc_dataset
""".strip()

bc_apply_cmd = """
python bin/production/run_exp5_workflow.py \
  --exp exp5 \
  --steps bc_apply
""".strip()

run_shell(bc_dataset_cmd, execute=EXECUTE_WORKFLOW_COMMANDS)
print()
run_shell(bc_apply_cmd, execute=EXECUTE_WORKFLOW_COMMANDS)


Suggested checks after these phases:

- do the bias-correction dataset files exist?
- do the corrected GCM files exist for the expected periods?
- does a quick comparison between raw and corrected fields look physically reasonable?

## 9. Training Or Checkpoint Reuse

Users have two options:

- reuse the published checkpoint when the setup remains compatible
- retrain if they change the domain, variables, preprocessing, normalization, or model setup

The conventional local location for published checkpoint bundles is:

- `scratch/checkpoint_bundles/`

Training should remain an explicit part of the notebook story, even if some
users follow the pretrained path first.


In [ ]:
train_cmd = """
python bin/production/run_exp5_workflow.py \
  --exp exp5 \
  --steps train \
  --test-name unet_all
""".strip()

run_shell(train_cmd, execute=EXECUTE_WORKFLOW_COMMANDS)


Suggested checks after training:

- does the run directory exist?
- was a checkpoint written?
- do the loss curves and summary metrics look stable enough to continue?

## 10. Prediction, Metrics, And Final Diagnostics

Scientific role:

- assess whether the downscaled results look physically and statistically credible

Technical role:

- generate predictions
- compute daily and monthly metrics
- compute VALUE-style summaries
- generate the main diagnostic figures

These final outputs should be shown directly in the notebook, not only written to disk.

In [ ]:
predict_cmd = """
python bin/production/run_exp5_workflow.py \
  --exp exp5 \
  --steps predict_loop \
  --test-name unet_all \
  --simu-test gcm_bc \
  --predict-start-date 20000101 \
  --predict-end-date 20141231
""".strip()

metrics_cmd = """
python bin/production/run_exp5_workflow.py \
  --exp exp5 \
  --steps metrics_day,metrics_month,value_metrics,plot_metrics_day,plot_metrics_month \
  --test-name unet_all \
  --simu-test gcm_bc \
  --predict-start-date 20000101 \
  --predict-end-date 20141231
""".strip()

run_shell(predict_cmd, execute=EXECUTE_WORKFLOW_COMMANDS)
print()
run_shell(metrics_cmd, execute=EXECUTE_WORKFLOW_COMMANDS)


In [ ]:
# Example inspection of locally available or downloaded Mercure outputs.
required_dir = COURSE_ASSETS_DIR / "required"

print(f"Checkpoint bundles are expected under: {CHECKPOINT_BUNDLES_DIR}")
print(f"Published course assets are searched under: {required_dir}")

daily_csv = required_dir / "metrics" / "metrics_test_mean_daily_exp5_unet_all_gcm_bc.csv"
monthly_csv = required_dir / "metrics" / "metrics_test_mean_monthly_exp5_unet_all_gcm_bc.csv"
value_csv = required_dir / "metrics" / "value_metrics_exp5_unet_all.csv"

show_csv(daily_csv)
show_csv(monthly_csv)
show_csv(value_csv)


In [ ]:
# Show precomputed published figures when available locally.
plot_names = [
    "daily_rmse_seasonal_unet_all_gcm_bc.png",
    "monthly_rmse_seasonal_unet_all_gcm_bc.png",
    "monthly_spatial_bias_distribution_unet_all_gcm_bc.png",
    "monthly_spatial_rmse_distribution_unet_all_gcm_bc.png",
    "exp5_pairwise_distribution_quantiles.png",
    "exp5_historical_5curve_pdf.png",
]

for name in plot_names:
    show_image(required_dir / "plots" / name)

Suggested final discussion:

- What do the prediction results look like scientifically?
- What do the bias diagnostics say?
- How should the VALUE metrics be interpreted?
- Which outputs were generated live, and which were reused from the published Mercure package?
- If the setup changes, which phases must be rerun and when is retraining necessary?

## 11. Summary

This notebook is intended to remain useful in two modes:

- as a readable guide explaining the science and the workflow
- as an executable document that users can rerun phase by phase on their own system

The key principle is transparency: one phase at a time, explicit checks after each phase, and clear discussion of the resulting statistics, tables, and figures.